# Round 3: the signs above written beside their letter

On the 100 real words nearly every error of every run was a sign that print places above
its letter: ꯥ dropped or read as ꯣ or ꯦ, ꯩ read as ꯧ, ꯪ read as ꯦ or ꯣ. The writer puts
these signs after the letter, at the top, where ꯣ and ꯦ stand (measured on those words:
ꯥ is centred 0.18 L past its letter's right edge; the synthetic words put it 0.3-0.4 L
before that edge, over the letter). Round 3 trains the final recipe of round 2 again with
half of the training words written that way (`--marks-beside 0.5`); nothing else changes.

1. **A look** at the new training words (TUMMHCD characters): the same words with the
   signs above as in print and beside, the first training words of the run, and TUMMHCD's
   own images of ꯥ, ꯩ and ꯪ. Look before training: if something is wrong, stop here.
2. **A fixed set with the signs beside**: the 5,000 validation words again, with every
   sign above written beside (`WORK/synth/val_beside.tar`), and the placement measured on
   the real words and on both sets. Writes `results/round3_sign_geometry.json` and
   `results/sign_placement_beside_tummhcd.json`.
3. **Round 2 on those words**: how much the new placement costs the present recogniser.
   Writes `results/round3_val_beside_<run>.json`.
4. **Training**, two runs of about 2.5 hours each (resumable, as in Phase 2).
5. **Validation**: the language model's weights chosen on the validation words, as in
   round 2; both validation sets read. Writes `results/phase2_val_<run>.json` and
   `results/round3_val_beside_<run>.json`.
6. **The 100 real words, as a development set**: they were round 2's test (used once;
   those results stand), and their errors shaped round 3, so they no longer test it. They
   show whether the change does what it was made for. Writes `results/round3_dev_<run>.json`.
7. **New real words, once** (`RUN_REAL2 = True`): the test of round 3 needs words its
   design has not seen: pages 7-12 of the writing pages (items 107-209), written as before
   and cut by Claude into `WORK/real/real_test2.tar`. Round 2 and round 3 read them once.
   Writes `results/phase3_real2_<run>.json`.
8. **Synthetic test, once** (`RUN_TEST = True`). Writes `results/phase2_test_<run>.json`.

Needs an A100 for section 4 (sections 1-3 and 5-8 run on any GPU). Inputs on Drive, from
the earlier notebooks: `WORK/glyphs`, `WORK/lexicon`, `WORK/synth/{val,test}.tar`,
`WORK/lm/char_lm.pkl`, `WORK/weights`, `WORK/runs/round2_*`, `WORK/results/phase2_val_*`
and `WORK/real/real_test.tar`. Afterwards run `collect_results.ipynb` and send the zip.

In [ ]:
# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-recognition"
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["claude/intelligent-euler-rx42yl", "main"]  # the first that has the round 3 code is used
HF_MODEL = "Chingkheinganba/handwritten-meitei-mayek-recognition"   # the first paper's weights, if not on Drive
STEPS = 60000       # training steps per run, 64 words each (as in round 2)
# Round 3 (owner, 27 September 2026: "create synthetic data which imitates the new patterns"): round 2's
# final recipe (ConvNeXt-T from TUMMHCD, 10% scrambled words, two seeds) with half the training words
# writing their signs above (ꯥ ꯩ ꯪ) after the letter at the top.
RUNS = {
    "round3_convnext_tummhcd": {"encoder": "convnext_tiny", "init": "tummhcd", "scrambled": 0.1,
                                "marks_beside": 0.5},
    "round3_convnext_tummhcd_seed1": {"encoder": "convnext_tiny", "init": "tummhcd", "scrambled": 0.1,
                                      "marks_beside": 0.5, "seed": 1, "data_seed": 1001},
}
ROUND2 = ["round2_convnext_tummhcd", "round2_convnext_tummhcd_seed1"]   # the same recipe without it
REAL_DEV = f"{WORK}/real/real_test.tar"    # the first 100 real words: now the development set
REAL2 = f"{WORK}/real/real_test2.tar"      # new real words (pages 7-12), once written and cut
RUN_REAL2 = False   # True once, when REAL2 is on Drive: round 2 and round 3 read the new words, one time
RUN_TEST = False    # True once, after every choice: round 3 reads the synthetic test set

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
url = f"https://github.com/{REPO}.git"
if not os.path.exists(f"{REPO_DIR}/scripts/measure_sign_geometry.py"):
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/scripts/measure_sign_geometry.py"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the round 3 code. git: " + r.stderr)
else:  # a runtime left from an earlier session: bring the code up to date (its outputs are on Drive)
    branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    run("git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", url, branch)
    run("git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", "timm", "huggingface_hub", "scipy")

needed = ["glyphs/train.npz", "glyphs/val.npz", "lexicon/train.tsv", "lexicon/val.tsv", "synth/val.tar",
          "synth/test.tar", "lm/char_lm.pkl", "real/real_test.tar"]
needed += [f"runs/{n}/best.pt" for n in ROUND2] + [f"results/phase2_val_{n}.json" for n in ROUND2]
missing = [f for f in needed if not os.path.exists(f"{WORK}/{f}")]
if missing:
    raise SystemExit("missing on Drive (from the Phase 1-3 notebooks):\n  " + "\n  ".join(missing))
for f in needed[:8]:
    os.makedirs(os.path.dirname(f"/content/{f}"), exist_ok=True)
    if not os.path.exists(f"/content/{f}"):
        shutil.copy(f"{WORK}/{f}", f"/content/{f}")   # local copies read faster than Drive
for d in ("results", "runs"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
WEIGHTS = "/content/weights"                  # the first paper's weights: start of the encoder
if not os.path.exists(WEIGHTS):
    if os.path.exists(f"{WORK}/weights"):
        shutil.copytree(f"{WORK}/weights", WEIGHTS)
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(HF_MODEL, local_dir=WEIGHTS)
        shutil.copytree(WEIGHTS, f"{WORK}/weights", dirs_exist_ok=True)
LM = "/content/lm/char_lm.pkl"
SIZES = "results/glyph_sizes_tummhcd.json"
VAL, VAL_BESIDE, TEST, DEV = ("/content/synth/val.tar", "/content/synth/val_beside.tar", "/content/synth/test.tar",
                              "/content/real/real_test.tar")
WORKERS = max((os.cpu_count() or 4) - 2, 2)   # processes rendering training words
LOOK = f"{WORK}/runs/round3_look"             # the pictures of section 1 (collected with the results)
os.makedirs(LOOK, exist_ok=True)

def train_run(name):
    """Trains run `name` of RUNS, or goes on with it (section 4)."""
    flags = [a for k, v in RUNS[name].items() for a in ("--" + k.replace("_", "-"), v)]
    stream(sys.executable, "-u", "scripts/train_recogniser.py", "--glyphs", "/content/glyphs/train.npz",
           "--lexicon", "/content/lexicon/train.tsv", "--sizes", SIZES, "--val", VAL,
           "--run-dir", f"{WORK}/runs/{name}", "--tummhcd-dir", WEIGHTS,
           "--results", f"{WORK}/results/phase2_train_{name}.json", "--steps", STEPS, "--workers", WORKERS, *flags)

def evaluate(name, data, out, tag, tune=False):
    """Run `name` reads a set: greedy, and with the language model at the weights chosen on validation
    (or chosen here, tune=True). Predictions and a sheet of misread words go to the run's folder."""
    choice = ["--tune"] if tune else ["--tuned", f"{WORK}/results/phase2_val_{name}.json"]
    run(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", f"{WORK}/runs/{name}/best.pt",
        "--set", data, "--lm", LM, *choice, "--out", out,
        "--predictions", f"{WORK}/runs/{name}/{tag}_predictions.tsv", "--sheet", f"{WORK}/runs/{name}/{tag}_errors.png")

def scores(files):
    """A table of results files: CER and WER in %, greedy and with the language model."""
    import pandas as pd
    rows = []
    for label, f in files:
        if os.path.exists(f):
            r = json.load(open(f))
            g, w = r["greedy"], r["with_lm"]
            rows.append({"": label, "words": w["words"], "CER": round(100 * g["cer"], 2), "WER": round(100 * g["wer"], 1),
                         "CER LM": round(100 * w["cer"], 2), "WER LM": round(100 * w["wer"], 1)})
    return pd.DataFrame(rows)

run("git", "log", "-1", "--format=%h %s")
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

## 1. A look at the new training words

Top: validation words with ꯥ, ꯩ or ꯪ, each drawn twice: left as print places the signs,
right as round 3 may (after the letter, at the top). Then the first 48 words the first run will see (before augmentation): about half
of the words with a sign above have it beside. Last: TUMMHCD's own images of the three
signs (the synthesiser draws each sign with these; only its place changes).

In [ ]:
import numpy as np
from IPython.display import Image as Show, display
from PIL import Image, ImageDraw, ImageFont
from mayek_words.glyphs import ASSETS, GlyphStore
from mayek_words.lexicon import Lexicon
from mayek_words.synth import Config, WordSynth, load_priors

ABOVE = "ꯥꯩꯪ"   # ꯥ ꯩ ꯪ
val_store = GlyphStore.load("/content/glyphs/val.npz")
priors = load_priors(sizes=SIZES)
font = ImageFont.truetype(str(ASSETS / "NotoSansMeeteiMayek-Regular.ttf"), 20, layout_engine=ImageFont.Layout.RAQM)
lex = Lexicon.load("/content/lexicon/val.tsv")
rng, texts = np.random.default_rng(0), []
while len(texts) < 16:
    w = lex.sample(rng)
    if any(c in ABOVE for c in w) and w not in texts:
        texts.append(w)

def tile(img, text, h=72):
    im = Image.fromarray(img)
    im = im.resize((max(round(im.width * h / im.height), 1), h))
    t = Image.new("L", (im.width + 16, h + 28), 255)
    t.paste(im, (0, 0))
    ImageDraw.Draw(t).text((4, h + 2), text, font=font, fill=0)
    return t

pairs = [[tile(WordSynth(val_store, priors, Config(p_marks_beside=p)).render(w, np.random.default_rng([5, k])).image, w)
          for p in (0.0, 1.0)] for k, w in enumerate(texts)]
cw = max(t.width for pr in pairs for t in pr) + 12
ch = max(t.height for pr in pairs for t in pr) + 6
sheet = Image.new("L", (4 * cw, 8 * ch), 255)
for k, (a, b) in enumerate(pairs):
    r, c = divmod(k, 2)
    sheet.paste(a, (2 * c * cw, r * ch))
    sheet.paste(b, (2 * c * cw + cw, r * ch))
sheet.save(f"{LOOK}/print_and_beside.png")
display(Show(f"{LOOK}/print_and_beside.png"))

run(sys.executable, "scripts/render_words.py", "--glyphs", "/content/glyphs/train.npz", "--lexicon",
    "/content/lexicon/train.tsv", "--sizes", SIZES, "--n", 48, "--seed", 1000, "--scrambled", 0.1,
    "--marks-beside", 0.5, "--sheet", f"{LOOK}/training_words.png")
display(Show(f"{LOOK}/training_words.png"))

rows = []
for ch_ in ABOVE:
    idx = val_store.candidates(ch_)
    pick = np.random.default_rng(1).choice(idx, size=min(16, len(idx)), replace=False)
    rows.append(np.concatenate([np.pad(val_store.gray[i], 2, constant_values=128) for i in pick], 1))
tiles = Image.fromarray(np.concatenate(rows, 0)).resize((16 * 28 * 3, 3 * 28 * 3), Image.NEAREST)
tiles.save(f"{LOOK}/tummhcd_signs_above.png")
display(Show(f"{LOOK}/tummhcd_signs_above.png"))

## 2. A fixed set with the signs beside, and where the signs stand

The 5,000 validation words of `synth/val.tar` again (the same words and word styles: same
seed), with every sign above written beside its letter (`--marks-beside 1`). Then the
signs' places measured on the ink of the real words, of `val.tar` and of the new set, and
the shapes of TUMMHCD's sign images; and the placement rules of the signs beside checked
for the new placement (never nearer the next letter, never touched by it).

In [ ]:
if not os.path.exists(f"{WORK}/synth/val_beside.tar"):
    run(sys.executable, "scripts/render_words.py", "--glyphs", "/content/glyphs/val.npz", "--lexicon",
        "/content/lexicon/val.tsv", "--sizes", SIZES, "--n", 5000, "--seed", 1, "--marks-beside", 1.0,
        "--out-dir", "/content/synth/val_beside")
    run("tar", "-cf", VAL_BESIDE, "-C", "/content/synth/val_beside", ".")
    shutil.copy(VAL_BESIDE, f"{WORK}/synth/val_beside.tar")
elif not os.path.exists(VAL_BESIDE):
    shutil.copy(f"{WORK}/synth/val_beside.tar", VAL_BESIDE)
run(sys.executable, "scripts/measure_sign_geometry.py", "--sets", f"real={DEV}", f"print={VAL}",
    f"beside={VAL_BESIDE}", "--glyphs", "/content/glyphs/val.npz", "--out", f"{WORK}/results/round3_sign_geometry.json")
run(sys.executable, "scripts/check_signs.py", "--glyphs", "/content/glyphs/val.npz", "--lexicon",
    "/content/lexicon/train.tsv", "--sizes", SIZES, "--marks-beside",
    "--out", f"{WORK}/results/sign_placement_beside_tummhcd.json")

## 3. How round 2 reads the signs beside

The final recipe of round 2 reads both validation sets (the same words; only the place of
the signs above differs), with the language model's weights chosen in Phase 2.

In [ ]:
for name in ROUND2:
    out = f"{WORK}/results/round3_val_beside_{name}.json"
    if not os.path.exists(out):
        evaluate(name, VAL_BESIDE, out, "val_beside")
display(scores([(f"{n} {s}", f"{WORK}/results/{p}_{n}.json") for n in ROUND2
                for s, p in (("print", "phase2_val"), ("beside", "round3_val_beside"))]))

## 4. Training

One cell per run; the first seed first. As in round 2 (60,000 steps of 64 words, the best
step on validation kept), with half of the training words writing the signs above beside.
A run saves its state every 1,000 steps to `WORK/runs/<run>`; if Colab disconnects, run the
first two cells and the run's cell again and it goes on from there.

In [ ]:
train_run("round3_convnext_tummhcd")

In [ ]:
train_run("round3_convnext_tummhcd_seed1")

## 5. Validation

Each round 3 run reads the validation words with the language model's weight (alpha) and
bonus per character (beta) chosen here, as in round 2, and then the words with the signs
beside at those weights. Round 2 for comparison.

In [ ]:
for name in RUNS:
    if not os.path.exists(f"{WORK}/runs/{name}/final.pt"):
        print(f"{name}: not trained yet")
        continue
    if not os.path.exists(f"{WORK}/results/phase2_val_{name}.json"):
        evaluate(name, VAL, f"{WORK}/results/phase2_val_{name}.json", "val", tune=True)
    if not os.path.exists(f"{WORK}/results/round3_val_beside_{name}.json"):
        evaluate(name, VAL_BESIDE, f"{WORK}/results/round3_val_beside_{name}.json", "val_beside")
display(scores([(f"{n} {s}", f"{WORK}/results/{p}_{n}.json") for n in ROUND2 + list(RUNS)
                for s, p in (("print", "phase2_val"), ("beside", "round3_val_beside"))]))

## 6. The 100 real words, as a development set

Round 3 reads the first 100 real words. Round 2's readings of them (its test, used once) are
in `phase3_real_<run>.json`; both are compared word by word, with the words holding a sign
above counted apart. These words shaped round 3, so this shows whether the change does what
it was made for; it is not a test.

In [ ]:
for name in RUNS:
    out = f"{WORK}/results/round3_dev_{name}.json"
    if not os.path.exists(f"{WORK}/results/phase2_val_{name}.json"):
        print(f"{name}: run sections 4 and 5 first")
    elif not os.path.exists(out):
        evaluate(name, DEV, out, "real_dev")
preds = [f"{n}={WORK}/runs/{n}/real_predictions.tsv" for n in ROUND2]
preds += [f"{n}={WORK}/runs/{n}/real_dev_predictions.tsv" for n in RUNS if os.path.exists(f"{WORK}/runs/{n}/real_dev_predictions.tsv")]
run(sys.executable, "scripts/compare_runs.py", "--predictions", *preds, "--focus", ABOVE, "ꯥ", "ꯩ", "ꯪ",
    "--out", f"{WORK}/results/round3_dev_comparison.json")
display(scores([(n, f"{WORK}/results/phase3_real_{n}.json") for n in ROUND2]
               + [(n, f"{WORK}/results/round3_dev_{n}.json") for n in RUNS]))

## 7. New real words, once

The test of round 3: words that neither round nor any choice has seen, by the same writer.
Write pages 7-12 of the writing pages (items 107-209) as before, send them to Claude, and
put the `real_test2.tar` you get back in `WORK/real/`. Then set `RUN_REAL2 = True` in the
first cell and run the first two cells and this one. Round 2 and round 3 (two seeds each)
read the words once; a run already read is not read again.

In [ ]:
if not RUN_REAL2:
    print("RUN_REAL2 is False: the new real words stay untouched")
elif not os.path.exists(REAL2):
    raise SystemExit(f"{REAL2} is not on Drive yet")
else:
    missing = [n for n in RUNS if not os.path.exists(f"{WORK}/results/phase2_val_{n}.json")]
    if missing:
        raise SystemExit(f"train and validate every round 3 run first (sections 4 and 5): {missing}")
    shutil.copy(REAL2, "/content/real/real_test2.tar")
    for name in ROUND2 + list(RUNS):
        out = f"{WORK}/results/phase3_real2_{name}.json"
        if os.path.exists(out):
            print(f"{name}: already read ({out}), not read again")
            continue
        evaluate(name, "/content/real/real_test2.tar", out, "real2")
    run(sys.executable, "scripts/compare_runs.py", "--predictions",
        *[f"{n}={WORK}/runs/{n}/real2_predictions.tsv" for n in ROUND2 + list(RUNS)],
        "--focus", ABOVE, "ꯥ", "ꯩ", "ꯪ", "--out", f"{WORK}/results/phase3_real2_comparison.json")
    display(scores([(n, f"{WORK}/results/phase3_real2_{n}.json") for n in ROUND2 + list(RUNS)]))

## 8. Synthetic test, once

Only after sections 5-7: round 3 reads the 5,000 synthetic test words once, with the
weights chosen on validation (round 2's test results are in `phase2_test_round2_*.json`).
Set `RUN_TEST = True` in the first cell, run the first two cells and this one.

In [ ]:
if not RUN_TEST:
    print("RUN_TEST is False: the test set stays untouched")
else:
    for name in RUNS:
        out = f"{WORK}/results/phase2_test_{name}.json"
        if os.path.exists(out):
            print(f"{name}: already tested ({out}), not tested again")
            continue
        run(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", f"{WORK}/runs/{name}/best.pt",
            "--set", TEST, "--lm", LM, "--tuned", f"{WORK}/results/phase2_val_{name}.json", "--out", out,
            "--predictions", f"{WORK}/runs/{name}/test_predictions.tsv")
    display(scores([(n, f"{WORK}/results/phase2_test_{n}.json") for n in ROUND2 + list(RUNS)]))

## Afterwards

Run `collect_results.ipynb` and send the zip: it holds the results files, the pictures of
section 1 (`runs/round3_look`), and each run's predictions and sheets of misread words.